In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_ROOT = PROJECT_ROOT / "data" / "raw"

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
QC_DIR = PROCESSED_ROOT / "qc"
QC_DIR.mkdir(parents=True, exist_ok=True)

QC_ALL_PATH = QC_DIR / "00_QC_ALL_SUBJECTS.csv"
IO_ALL_PATH = QC_DIR / "00_IO_ALL.csv"

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_ROOT     :", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("QC_DIR       :", QC_DIR, "| exists:", QC_DIR.exists())
print("QC_ALL_PATH  :", QC_ALL_PATH)
print("IO_ALL_PATH  :", IO_ALL_PATH)


In [ ]:
import sys, inspect, importlib

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import trigno_io
importlib.reload(trigno_io)
from trigno_io import read_trigno_csv

print("read_trigno_csv imported from:", inspect.getfile(read_trigno_csv))


In [ ]:
import numpy as np
import pandas as pd

def estimate_fs_and_gaps(t, gap_factor=2.0):
    t = np.asarray(t, dtype=float)
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) == 0:
        return np.nan, 0, np.nan, np.nan
    fs = 1.0 / np.mean(dt)
    med_dt = np.median(dt)
    gap_count = int(np.sum(dt > gap_factor * med_dt))
    return fs, gap_count, float(np.min(dt)), float(np.max(dt))

def nan_ratio(x):
    x = np.asarray(x)
    return float(np.mean(~np.isfinite(x)))

def near_zero_variance(x, eps=1e-10):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return True, np.nan
    v = float(np.var(x))
    return (v < eps), v

def saturation_ratio(x, run_len_samples=500, tol=0.0):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    x = x[mask]
    if len(x) == 0:
        return np.nan

    if tol == 0.0:
        same = np.diff(x) == 0
    else:
        same = np.abs(np.diff(x)) <= tol

    total_marked = 0
    current = 1
    for s in same:
        if s:
            current += 1
        else:
            if current >= run_len_samples:
                total_marked += current
            current = 1
    if current >= run_len_samples:
        total_marked += current

    return float(total_marked / max(len(x), 1))

def qc_one_signal(t, x, name_prefix, gap_factor=2.0, sat_run=500, sat_tol=0.0, var_eps=1e-10):
    fs, gap_count, dt_min, dt_max = estimate_fs_and_gaps(t, gap_factor=gap_factor)
    nratio = nan_ratio(x)
    is_lowvar, var = near_zero_variance(x, eps=var_eps)
    sratio = saturation_ratio(x, run_len_samples=sat_run, tol=sat_tol)

    t = np.asarray(t, dtype=float)
    t0 = float(np.nanmin(t)) if np.any(np.isfinite(t)) else np.nan
    t1 = float(np.nanmax(t)) if np.any(np.isfinite(t)) else np.nan
    dur = float(t1 - t0) if (np.isfinite(t0) and np.isfinite(t1)) else np.nan

    return {
        f"{name_prefix}_fs_est": fs,
        f"{name_prefix}_gaps": gap_count,
        f"{name_prefix}_dt_min": dt_min,
        f"{name_prefix}_dt_max": dt_max,
        f"{name_prefix}_nan_ratio": nratio,
        f"{name_prefix}_var": var,
        f"{name_prefix}_lowvar_flag": bool(is_lowvar),
        f"{name_prefix}_saturation_ratio": sratio,
        f"{name_prefix}_duration_s": dur,
        f"{name_prefix}_n_samples": int(np.sum(np.isfinite(t))),
    }


In [ ]:
import numpy as np
import pandas as pd

def _nan_qc(prefix: str):
    return {
        f"{prefix}_fs_est": np.nan,
        f"{prefix}_gaps": np.nan,
        f"{prefix}_dt_min": np.nan,
        f"{prefix}_dt_max": np.nan,
        f"{prefix}_nan_ratio": np.nan,
        f"{prefix}_var": np.nan,
        f"{prefix}_lowvar_flag": False,
        f"{prefix}_saturation_ratio": np.nan,
        f"{prefix}_duration_s": np.nan,
        f"{prefix}_n_samples": 0,
    }

def qc_trigno_parsed(per_sensor: dict):
    rows = []
    for sensor, d in per_sensor.items():
        emg = d.get("emg", None)
        imu = d.get("imu", None)

        r = {"sensor": sensor}

        # EMG QC
        if emg is None or len(emg) == 0:
            r.update(_nan_qc("emg"))
        else:
            r.update(qc_one_signal(
                t=emg["t_emg"].values,
                x=emg["emg_mv"].values,
                name_prefix="emg",
                gap_factor=2.0,
                sat_run=500,
                sat_tol=0.0,
                var_eps=1e-10
            ))

        # IMU QC on magnitudes
        if imu is None or len(imu) == 0:
            r.update(_nan_qc("imu_gyro_mag"))
            r.update(_nan_qc("imu_acc_mag"))
        else:
            acc_mag = np.sqrt(
                imu["acc_x_g"].values**2 + imu["acc_y_g"].values**2 + imu["acc_z_g"].values**2
            )
            gyro_mag = np.sqrt(
                imu["gyro_x_dps"].values**2 + imu["gyro_y_dps"].values**2 + imu["gyro_z_dps"].values**2
            )

            r.update(qc_one_signal(
                t=imu["t_imu"].values,
                x=gyro_mag,
                name_prefix="imu_gyro_mag",
                gap_factor=2.0,
                sat_run=60,
                sat_tol=0.0,
                var_eps=1e-10
            ))

            r.update(qc_one_signal(
                t=imu["t_imu"].values,
                x=acc_mag,
                name_prefix="imu_acc_mag",
                gap_factor=2.0,
                sat_run=60,
                sat_tol=0.0,
                var_eps=1e-10
            ))

        rows.append(r)

    return pd.DataFrame(rows)



In [ ]:
from pathlib import Path

def subject_csv_files(subject_name: str) -> list[Path]:
    subject_dir = RAW_ROOT / subject_name / "EMG&IMUTest"
    if not subject_dir.exists():
        raise FileNotFoundError(f"Subject dir not found: {subject_dir}")
    return sorted(subject_dir.glob("*.csv"))


In [ ]:
def qc_and_meta_for_subject(subject_name: str, files: list[Path]):
    qc_rows = []
    meta_rows = []

    for fp in files:
        sensors, per_sensor, meta = read_trigno_csv(fp)

        qc_df = qc_trigno_parsed(per_sensor)
        qc_df.insert(0, "trial_id", fp.stem)
        qc_df.insert(0, "file", fp.name)
        qc_df.insert(0, "subject", subject_name)

        # attach layout + mapping info to each QC row (helps later debugging)
        qc_df["layout"] = meta.get("layout", "")
        qc_df["mapping_ok"] = meta.get("mapping_ok", True)

        qc_rows.append(qc_df)

        meta_rows.append({
            "subject": subject_name,
            "trial_id": fp.stem,
            "file": fp.name,

            "layout": meta.get("layout", ""),
            "block_cols": meta.get("block_cols", np.nan),
            "nsensors": meta.get("nsensors", np.nan),

            "ncols_available_in_header": meta.get("ncols_available_in_header", np.nan),
            "ncols_requested_usecols": meta.get("ncols_requested_usecols", np.nan),
            "ncols_read_after_drop_empty": meta.get("ncols_read_after_drop_empty", np.nan),
            "ncols_used": meta.get("ncols_used", np.nan),
            "nrows_loaded": meta.get("nrows_loaded", np.nan),

            "sensor_name_source": meta.get("sensor_name_source", ""),
            "mapping_ok": meta.get("mapping_ok", True),
            "mapping_issues": meta.get("mapping_issues", ""),
            "final_sensor_names": "|".join(meta.get("final_sensor_names", [])) if meta.get("final_sensor_names") else "",
        })

    qc_subject_df = pd.concat(qc_rows, ignore_index=True) if qc_rows else pd.DataFrame()
    io_subject_df = pd.DataFrame(meta_rows)
    return qc_subject_df, io_subject_df


In [ ]:
def append_subject_to_master(df_subject: pd.DataFrame, subject_name: str, master_path: Path, key_col="subject"):
    if df_subject.empty:
        print(f"No rows for {subject_name}. Nothing to append -> {master_path.name}")
        return

    if master_path.exists():
        existing = pd.read_csv(master_path)
        already_done = subject_name in set(existing[key_col].astype(str))
    else:
        already_done = False

    if already_done:
        print(f"Subject '{subject_name}' already exists in {master_path.name}. Skipping append.")
        return

    write_header = not master_path.exists()
    df_subject.to_csv(master_path, mode="a", header=write_header, index=False)
    print(f"Appended '{subject_name}' -> {master_path}")


healthy 1

In [ ]:
SUBJECT_NAME = "Healthy_Subject_1"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_1"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 2

In [ ]:
SUBJECT_NAME = "Healthy_Subject_2"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_2"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 3

In [ ]:
SUBJECT_NAME = "Healthy_Subject_3"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_3"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 4

In [ ]:
SUBJECT_NAME = "Healthy_Subject_4"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_4"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 5

In [ ]:
SUBJECT_NAME = "Healthy_Subject_5"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_5"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 6

In [ ]:
SUBJECT_NAME = "Healthy_Subject_6"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_6"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 7

In [ ]:
SUBJECT_NAME = "Healthy_Subject_7"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")
append_subject_to_master(io_subject_df, SUBJECT_NAME, IO_ALL_PATH, key_col="subject")


In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_7"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")


healthy 8 

In [ ]:
SUBJECT_NAME = "Healthy_Subject_8"  # change only this each run

files = subject_csv_files(SUBJECT_NAME)
print("Files:", len(files))
for f in files:
    print(" -", f.name)

qc_subject_df, io_subject_df = qc_and_meta_for_subject(SUBJECT_NAME, files)

print("\nQC rows:", len(qc_subject_df))
display(qc_subject_df.head(10))

print("\nIO rows:", len(io_subject_df))
display(io_subject_df)

append_subject_to_master(qc_subject_df, SUBJECT_NAME, QC_ALL_PATH, key_col="subject")

In [ ]:
import pandas as pd
import numpy as np

SUBJECT_NAME = "Healthy_Subject_8"  # change only this each run

GAP_THRESHOLD = 0
SAT_THRESHOLD = 0.01

EXPECTED_TRIALS = 8
EXPECTED_SENSORS = 8
EXPECTED_ROWS = EXPECTED_TRIALS * EXPECTED_SENSORS

qc_all = pd.read_csv(QC_ALL_PATH)
qc_subj = qc_all[qc_all["subject"] == SUBJECT_NAME].copy()

print(f"Subject: {SUBJECT_NAME}")
print(f"Total QC rows: {len(qc_subj)}")
print(f"Trials: {qc_subj['trial_id'].nunique()} | Sensors: {qc_subj['sensor'].nunique()}")

# completeness checks (note: EMG-only / IMU-only still should have 8 sensors if file has 8 sensors)
if qc_subj['trial_id'].nunique() != EXPECTED_TRIALS:
    print(f"⚠️ Expected {EXPECTED_TRIALS} trials but found {qc_subj['trial_id'].nunique()}")
if qc_subj['sensor'].nunique() != EXPECTED_SENSORS:
    print(f"⚠️ Expected {EXPECTED_SENSORS} sensors but found {qc_subj['sensor'].nunique()}")
if len(qc_subj) != EXPECTED_ROWS:
    print(f"⚠️ Expected {EXPECTED_ROWS} QC rows but found {len(qc_subj)}")

# IMPORTANT: fillna(0) so missing modality doesn't trigger gaps
qc_subj["qc_fail"] = (
    (qc_subj["emg_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_gyro_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["imu_acc_mag_gaps"].fillna(0) > GAP_THRESHOLD) |
    (qc_subj["emg_lowvar_flag"].fillna(False) == True) |
    (qc_subj["emg_saturation_ratio"].fillna(0) > SAT_THRESHOLD) |
    (qc_subj["mapping_ok"].fillna(True) == False)
)

print("\nQC FAIL rows:", int(qc_subj["qc_fail"].sum()))

trial_status = qc_subj.groupby("trial_id")["qc_fail"].any().reset_index()
trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")

print("\n=== Trial QC status ===")
display(trial_status.sort_values("trial_id"))

if qc_subj["qc_fail"].any():
    print("\n=== QC FAIL details (rows) ===")
    display(
        qc_subj.loc[qc_subj["qc_fail"], [
            "trial_id","sensor","layout","mapping_ok",
            "emg_gaps","imu_gyro_mag_gaps","imu_acc_mag_gaps",
            "emg_lowvar_flag","emg_var","emg_saturation_ratio"
        ]].sort_values(["trial_id","sensor"])
    )
else:
    print("\n✓ No QC failures detected under current thresholds.")

# FS & duration summary (safe on NaNs)
summary = pd.DataFrame({
    "metric": [
        "emg_fs_est", "imu_gyro_mag_fs_est", "imu_acc_mag_fs_est",
        "emg_duration_s", "imu_gyro_mag_duration_s"
    ],
    "min": [
        qc_subj["emg_fs_est"].min(),
        qc_subj["imu_gyro_mag_fs_est"].min(),
        qc_subj["imu_acc_mag_fs_est"].min(),
        qc_subj["emg_duration_s"].min(),
        qc_subj["imu_gyro_mag_duration_s"].min(),
    ],
    "median": [
        qc_subj["emg_fs_est"].median(),
        qc_subj["imu_gyro_mag_fs_est"].median(),
        qc_subj["imu_acc_mag_fs_est"].median(),
        qc_subj["emg_duration_s"].median(),
        qc_subj["imu_gyro_mag_duration_s"].median(),
    ],
    "max": [
        qc_subj["emg_fs_est"].max(),
        qc_subj["imu_gyro_mag_fs_est"].max(),
        qc_subj["imu_acc_mag_fs_est"].max(),
        qc_subj["emg_duration_s"].max(),
        qc_subj["imu_gyro_mag_duration_s"].max(),
    ],
})
print("\n=== FS & DURATION SUMMARY (min / median / max) ===")
display(summary)

# IO meta anomalies (file-level)
if IO_ALL_PATH.exists():
    io_all = pd.read_csv(IO_ALL_PATH)
    io_subj = io_all[io_all["subject"] == SUBJECT_NAME].copy()

    print("\n=== IO META (per trial) ===")
    display(io_subj.sort_values("trial_id"))

    print("\n=== IO anomalies (mapping_ok False OR nsensors != 8) ===")
    anomalies = io_subj[(io_subj["mapping_ok"] == False) | (io_subj["nsensors"] != EXPECTED_SENSORS)]
    if anomalies.empty:
        print("✓ No IO anomalies detected.")
    else:
        display(anomalies.sort_values("trial_id"))
else:
    print("\n(IO meta file not found: 00_IO_ALL.csv) — skipping IO anomaly checks.")
